# **Libraries**


In [9]:
from langchain_neo4j import LLMGraphTransformer #  This class is used to extract nodes and relationships from text
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate # It can define system and user messages separately.
from langchain_community.document_loaders import PyPDFLoader # Used to load the PDF document to convert it later to document objects
from langchain_text_splitters import RecursiveCharacterTextSplitter # Used for splitting the documents into smaller chunks (Piece of text) for better processing and analysis

import ollama

from pyvis.network import Network
import os

import torch

# **Loading the documents**


In [10]:
doc_path = r"/home/mbm/GraphRag/abel-and-his-great-adventure.pdf"

if doc_path: # Checking if the document path exist
    # Initializing the loader
    loader = PyPDFLoader(file_path=doc_path) # Loading the PDF document using PyPDFLoader and creating the document objects
    
    pages = loader.load() #  The pages is a List of document objects, where each document object represents a page in the PDF. 
    
    print(f"Loaded {len(pages)} document object from {doc_path}")
else:
    print("No document path provided.")

# Access content
print(len(pages))
#content = data[0].page_content # Reading the first page of the document 
#print(content[:]) # printing the content of the first page

Loaded 11 document object from /home/mbm/GraphRag/abel-and-his-great-adventure.pdf
11


# **KGGEN**


In [11]:
from kg_gen import KGGen

In [12]:
kg = KGGen(model="ollama/gpt-oss:20b")

In [13]:
from kg_gen import Graph

story_text = "\n\n".join(page.page_content for page in pages)  # KGGen expects a plain string, not Document objects

CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

# Steers extraction away from literal triple-mining of figurative prose (metaphors, imagery,
# abstract nouns like "soul" / "origin of evil") and toward the things we actually want in the
# graph: characters, places, animals, objects and the events/relations between them.
EXTRACTION_CONTEXT = (
    "This text is a work of literary fiction, not a factual or technical document. "
    "Extract only concrete entities: named people, animals, places, and physical objects, "
    "and the concrete events or relationships between them (e.g. who lives where, who owns "
    "what, who talks to whom, family relations, etc..). Resolve pronouns (he/she/I/you) to the "
    "character's name when it is clear from context. Do NOT extract abstract nouns, emotions, "
    "metaphors, or descriptive imagery (e.g. 'soul', 'origin of evil', 'warm colours') as entities."
)

# Chunking manually and looping sequentially, instead of kg.generate(chunk_size=...): that path
# fires every chunk concurrently via ThreadPoolExecutor, which just queues up against a single
# local Ollama GPU instance and times out. One request at a time matches how Ollama actually
# processes them, so nothing is wasted waiting.
chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
text_chunks = chunk_splitter.split_text(story_text)
print(f"Split into {len(text_chunks)} chunks")

all_entities = set()
all_relations = set()

for i, chunk in enumerate(text_chunks):
    # gpt-oss:20b occasionally returns an empty response (reasoning-model quirk) and dspy raises
    # instead of returning an empty graph. Skip that one chunk rather than losing everything
    # accumulated so far in the loop.
    try:
        chunk_graph = kg.generate(input_data=chunk, context=EXTRACTION_CONTEXT, cluster=False)
    except Exception as e:
        print(f"Chunk {i + 1}/{len(text_chunks)} failed, skipping: {e}")
        continue

    all_entities.update(chunk_graph.entities)
    all_relations.update(chunk_graph.relations)
    print(f"Chunk {i + 1}/{len(text_chunks)} done | {len(all_entities)} entities so far")

graph = Graph(
    entities=all_entities,
    relations=all_relations,
    edges={r[1] for r in all_relations},
)

print(f"Entities: {len(graph.entities)}, Relations: {len(graph.relations)}")

Split into 11 chunks
Chunk 1/11 done | 14 entities so far
Chunk 2/11 done | 44 entities so far
Chunk 3/11 done | 55 entities so far
Chunk 4/11 done | 67 entities so far
Chunk 5/11 done | 76 entities so far
Chunk 6/11 done | 86 entities so far
Chunk 7/11 done | 108 entities so far
Chunk 8/11 done | 136 entities so far
Chunk 9/11 done | 143 entities so far
Chunk 10/11 done | 149 entities so far
Chunk 11/11 done | 159 entities so far
Entities: 159, Relations: 135


In [14]:
from dspy.utils.exceptions import AdapterParseError

MAX_CLUSTER_ATTEMPTS = 5

# Same gpt-oss:20b empty-response quirk as above, but kg.cluster() runs many LLM calls internally
# (extract/validate/choose-representative per batch) with no retry of its own, so one flaky call
# aborts the whole merge. Kept in its own cell so a failure here doesn't force re-running the
# ~minutes-long extraction loop above — just re-run this cell.
for attempt in range(1, MAX_CLUSTER_ATTEMPTS + 1):
    try:
        clustered_graph = kg.cluster(graph, context=EXTRACTION_CONTEXT)  # merges duplicate/coreferent entities
        graph = clustered_graph
        break
    except AdapterParseError as e:
        print(f"Cluster attempt {attempt}/{MAX_CLUSTER_ATTEMPTS} failed, retrying: {e}")
else:
    print("Clustering failed after all retries; continuing with the unclustered graph.")

print(f"Entities: {len(graph.entities)}, Relations: {len(graph.relations)}")

Cluster attempt 1/5 failed, retrying: The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response:  

Expected to find output fields in the LM response: [representative] 


Cluster attempt 2/5 failed, retrying: The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response:  

Expected to find output fields in the LM response: [representative] 


Cluster attempt 3/5 failed, retrying: The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response:  

Expected to find output fields in the LM response: [representative] 


Cluster attempt 4/5 failed, retrying: The LM returned an empty or null response.

Adapter JSONAdapter failed to parse the LM response. 

LM Response:  

Expected to find output fields in the LM response: [representative] 


Cluster attempt 5/5 failed, retrying: The LM returned an empty or null response.

Adapter JSONAdapter fa

In [15]:
import json

nodes_data = [{"id": e, "type": "Entity", "properties": {}} for e in graph.entities]

with open("KGGEN2nodes.json", "w") as f:
    json.dump(nodes_data, f, indent=2)

# Relationships: (subject, predicate, object) tuples -> Cypher-friendly dicts
def to_rel_type(predicate):
    return predicate.upper().replace(" ", "_")

relationships_data = [
    {"source": s, "target": o, "type": to_rel_type(p), "properties": {}}
    for s, p, o in graph.relations
]

with open("KGGEN2relationships.json", "w") as f:
    json.dump(relationships_data, f, indent=2)

print(f"Exported {len(nodes_data)} nodes to KGGENnodes.json")
print(f"Exported {len(relationships_data)} relationships to KGGENrelationships.json")

if graph.entity_clusters:
    print("\nEntity clusters (canonical -> merged variants):")
    for canonical, variants in graph.entity_clusters.items():
        print(f"  {canonical}: {variants}")

Exported 159 nodes to KGGENnodes.json
Exported 135 relationships to KGGENrelationships.json
